# **Coding Exercise : #0803**
Parameter-Efficient Fine-Tuning (PEFT) : LoRA Fine-tuning(Exercise)


## **0. Import Library**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from google.colab import files

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import load_dataset, Dataset
from peft import get_peft_model, LoraConfig, TaskType

from sklearn.model_selection import train_test_split
from sklearn.metrics import *

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## **1. Load dataset**
* Uploading emotions.csv

In [ ]:
uploaded = files.upload()
data_path = list(uploaded.keys())[0]

In [ ]:
data_path

In [ ]:
data = pd.read_csv(data_path)
data.head()

In [ ]:
sns.countplot(x='label_name', data = data, hue = 'label_name')
plt.grid()
plt.show()

In [ ]:
data['word_count'] = data['text'].apply(lambda x: len(x))
data.head()

In [ ]:
sns.histplot(x='word_count', data = data, bins = 50)
plt.grid()
plt.show()

In [ ]:
labels = data.loc[:, ['label', 'label_name']].drop_duplicates()
labels = labels.sort_values(by = 'label')
print(labels)

label_list = labels['label_name'].tolist()
print(label_list)

## **2. Pre-processing Data**
* save the data with the required columns and split it into train and validation sets.
* convert the data into a tensor dataset structure to be used for training.

## **3. Tokenization**
* download the tokenizer for the model you will use.
* tokenize the dataset’s text column(set max_length = 200)


## **4. Load Pre-trained Model**
* Now, download the BERT model. At this point, set num_labels = 6 to add an output layer to the BERT model.


## **5. Set Training Arguments**
* Configure the hyperparameters for model training
* The LoRA fine-tuning procedure is identical to regular fine-tuning except for the LoRA configuration step.
* In the LoRA config, set the required hyperparameters and combine them with the base model using get_peft_model() to create the LoRA-enabled model.


In [ ]:
# LoRA configuration


# Apply LoRA (train only adapter layers)



## **6. Fine-tune Model**
* The LoRA fine-tuning process is the same as standard fine-tuning except for the LoRA configuration step.
* Simply specify the LoRA model prepared earlier when creating the Trainer, and proceed with training as usual.



## **7. Evaluate**
* Make predictions on the validation dataset and evaluate the classification model.
* For evaluating the classification model, we use scikit-learn’s confusion_matrix and classification_report functions.
* To use these functions, the predictions and labels must be converted to NumPy arrays.

In [ ]:
want_keys = ["input_ids", "attention_mask"]              # keys the model expects (add "token_type_ids" for BERT if present)
batch = {k: torch.tensor(val_ds[k]) for k in want_keys}  # convert dataset columns (lists) into torch tensors
batch = {k: v.to(device) for k, v in batch.items()}      # move tensors to the target device (CPU/GPU)

with torch.no_grad():                                    # disable gradient tracking for inference
    outputs = model(**batch)                             # forward pass; batch must be a dict of tensors
logits = outputs.logits                                  # model scores, shape: [N, num_labels]
pred  = logits.argmax(dim=-1)                            # predicted class ids (argmax over label dimension)
pred = pred.cpu().numpy()

In [ ]:
pred = pred.cpu().numpy()
print(confusion_matrix(val_ds['label'], pred))
print(classification_report(val_ds['label'], pred, target_names = label_list))